# 1. Data Cleaning and Preparation

The rainfall data for Machakos County was provided in raw text format, containing some invalid and missing values. The data needs to be cleaned before it can be used for numerical analysis.

A Python function using `try` and `except ValueError` is used to convert valid rainfall readings into integers. Invalid values are replaced with the rounded average of the valid Machakos rainfall readings. The months with imputed values are also identified and printed.

This process ensures that the Machakos data is consistent with the rainfall data from the other four counties.

In [2]:
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

In [5]:
def clean_machakos(data):
    valid_values = []
    cleaned_values = []
    imputed_months = []

    for i, value in enumerate(data):
        try:
            rainfall = int(value.strip())
            valid_values.append(rainfall)
            cleaned_values.append(rainfall)

        except ValueError:
            cleaned_values.append(None)
            imputed_months.append(months[i])

    average = round(np.mean(valid_values))

    cleaned_values = [
        average if value is None else value
        for value in cleaned_values
    ]

    print("Average rainfall used for imputation:", average, "mm")
    print("Months imputed:", imputed_months)
    print("Cleaned Machakos rainfall:", cleaned_values)

    return cleaned_values

machakos_clean = clean_machakos(machakos_raw)

Average rainfall used for imputation: 68 mm
Months imputed: ['April', 'September']
Cleaned Machakos rainfall: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


# 2. Combining Data into a NumPy Array

After cleaning the Machakos rainfall data, the rainfall records for all five counties are combined into a single NumPy array.

The array contains five rows representing the counties and twelve columns representing the months from January to December.

The shape and data type of the array are displayed to confirm that the data has been correctly organized for analysis.

In [6]:
# Add the cleaned Machakos data to the other four counties
all_rows = clean_rows + [machakos_clean]

# Convert the data into a NumPy array
R = np.array(all_rows)

# Display the rainfall array
print("Rainfall Array:")
print(R)

# Display shape and data type
print("Array shape:", R.shape)
print("Array data type:", R.dtype)

Rainfall Array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array data type: int64


# 3. Annual Rainfall Totals and Monthly Averages

This section analyses the rainfall distribution across the five counties.

NumPy's `sum()` function is used to calculate the total annual rainfall for each county, while the `mean()` function calculates the average rainfall received in each month across all five counties.

The results help identify differences in annual rainfall between counties and show how rainfall is distributed throughout the year.

In [7]:
# Calculate annual rainfall for each county
annual_totals = np.sum(R, axis=1)

print("Annual Rainfall Totals:")

for county, total in zip(counties, annual_totals):
    print(county, ":", total, "mm")

Annual Rainfall Totals:
Nairobi : 1045 mm
Nakuru : 916 mm
Kisumu : 1371 mm
Mombasa : 1075 mm
Machakos : 811 mm


In [8]:
# Calculate average rainfall for each month
monthly_averages = np.mean(R, axis=0)

print("Average Monthly Rainfall Across All Counties:")

for month, average in zip(months, monthly_averages):
    print(month, ":", round(average, 2), "mm")

Average Monthly Rainfall Across All Counties:
January : 47.0 mm
February : 47.6 mm
March : 91.0 mm
April : 164.4 mm
May : 171.2 mm
June : 67.6 mm
July : 56.6 mm
August : 58.2 mm
September : 64.2 mm
October : 67.8 mm
November : 127.4 mm
December : 80.6 mm


# 4. Identifying the Wettest Month for Each County

This section identifies the month with the highest rainfall in each county.

NumPy's `argmax()` function is applied along axis 1 to find the position of the highest rainfall value in each county's row. The `zip()` function is then used to pair each county with its corresponding wettest month.

The results help identify the periods when each county receives the highest amount of rainfall.

In [9]:
# Find the position of the highest rainfall in each county
wettest_indices = np.argmax(R, axis=1)

# Pair counties with their wettest months using zip()
print("Wettest Month for Each County:")

for county, index in zip(counties, wettest_indices):
    print(county, ":", months[index])

Wettest Month for Each County:
Nairobi : April
Nakuru : April
Kisumu : April
Mombasa : May
Machakos : November


# 5. Identifying and Counting Dry Months

A dry month is defined in this analysis as a month receiving less than 50 mm of rainfall.

A Boolean mask is created using the condition `R < 50` to identify months that meet this definition. NumPy's `sum()` function is then used to count the number of dry months for each county.

This analysis helps identify counties that experience more months with relatively low rainfall and may require additional water management measures.

In [10]:
# Create a Boolean mask for rainfall below 50 mm
dry_mask = R < 50

# Count the dry months for each county
dry_month_counts = np.sum(dry_mask, axis=1)

print("Number of Dry Months per County:")

for county, count in zip(counties, dry_month_counts):
    print(county, ":", count, "dry months")

Number of Dry Months per County:
Nairobi : 4 dry months
Nakuru : 3 dry months
Kisumu : 0 dry months
Mombasa : 2 dry months
Machakos : 6 dry months


# 6. Identifying the Best Two-Month Planting Windows

This section identifies the two consecutive months with the highest combined rainfall for each county.

The shifted-array technique is used by adding `R[:, :-1]` and `R[:, 1:]` to calculate the combined rainfall for every consecutive two-month period.

NumPy's `argmax()` function identifies the two-month period with the highest combined rainfall in each county. The resulting period is considered a potential planting window because it receives the highest combined rainfall compared to other consecutive two-month periods.

These findings can help guide farmers when planning their planting activities.

In [14]:
# Add rainfall for each month to the rainfall of the following month
two_month_rainfall = R[:, :-1] + R[:, 1:]

print("Combined Rainfall for Consecutive Months:")
print(two_month_rainfall)

Combined Rainfall for Consecutive Months:
[[120 148 311 395 211  53  42  55  92 226 269]
 [ 77 112 215 265 198 174 201 177 127 115  98]
 [148 240 390 420 275 165 155 175 178 218 225]
 [ 45  70 205 390 350 200 155 125 150 190 170]
 [ 83 123 156 208 160  29  21  80 113 227 278]]


In [16]:
# Find the position with the highest combined rainfall
best_window_indices = np.argmax(two_month_rainfall, axis=1)

# Display the recommended planting windows
print("Recommended Planting Windows:")

for county, index in zip(counties, best_window_indices):
    window = months[index] + " - " + months[index + 1]
    rainfall = two_month_rainfall[counties.index(county), index]

    print(county, ":", window, "(", rainfall, "mm )")

Recommended Planting Windows:
Nairobi : April - May ( 395 mm )
Nakuru : April - May ( 265 mm )
Kisumu : April - May ( 420 mm )
Mombasa : April - May ( 390 mm )
Machakos : November - December ( 278 mm )


# 7. Exporting the Rainfall Summary to CSV

This section exports the rainfall analysis results into a CSV file named `rainfall_summary.csv`.

The file contains the county names, annual rainfall totals, wettest months, number of dry months and recommended two-month planting windows.

Python's built-in `csv` module is used to create the file. This makes it easier to store, share and review the rainfall analysis results.

In [18]:

import csv

# Create the CSV file
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    # Write the column headings
    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    # Write the results for all five counties
    for county, total, wet_index, dry_count, window_index in zip(
        counties,
        annual_totals,
        wettest_indices,
        dry_month_counts,
        best_window_indices
    ):

        planting_window = (
            months[window_index] + " - " + months[window_index + 1]
        )

        writer.writerow([
            county,
            int(total),
            months[wet_index],
            int(dry_count),
            planting_window
        ])

print("rainfall_summary.csv created successfully!")


rainfall_summary.csv created successfully!


In [20]:

# Read and display the CSV file
with open("rainfall_summary.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)

    for row in reader:
        print(row)


['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April - May']
['Nakuru', '916', 'April', '3', 'April - May']
['Kisumu', '1371', 'April', '0', 'April - May']
['Mombasa', '1075', 'May', '2', 'April - May']
['Machakos', '811', 'November', '6', 'November - December']


In [23]:
from google.colab import files
files.download("rainfall_summary.csv")## Planting Advisory


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Planting Advisory

The rainfall analysis indicates that April and May are the most suitable potential planting window for Nairobi, Nakuru, Kisumu and Mombasa because they have the highest combined rainfall during these consecutive months. In Machakos, November and December record the highest combined rainfall, making this the recommended potential planting window. Farmers should prepare their land before the identified planting periods to take advantage of the available rainfall. The NGO should use SMS messages to inform farmers about the recommended planting periods and encourage them to monitor local weather forecasts before planting. Farmers should also consider soil conditions, crop water requirements and access to water when making planting decisions. These recommendations are based on historical monthly rainfall data and do not guarantee future rainfall patterns.

## Data Quality Notes

The Machakos rainfall data was initially provided as raw text and contained two invalid entries: "N/A" for April and an empty value for September. These missing values were replaced using the rounded average of the valid Machakos rainfall readings, which was 68 mm. Although this method allows the data to be used for numerical analysis, the imputed values are estimates and may differ from the actual rainfall recorded during those months. The analysis assumes that the other four counties' rainfall readings are accurate because no errors were identified in their supplied data. The NGO should verify the original Machakos weather station records before using the findings for long-term agricultural planning.